# Ranking 实验结果可视化与表格生成
此脚本会自动读取 `ranking/` 文件夹下的所有 JSON 结果文件，计算均值和 95% 置信区间，并生成对比表格。

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# 寻找所有的 ranking 结果文件
files = glob.glob('ranking/*.json')
if not files:
    print("在 ranking/ 文件夹下没有找到 JSON 文件。请先运行 exp1_run_parallel_ranking.ipynb")

for filename in files:
    with open(filename, 'r', encoding='utf-8') as f:
        data = json.load(f)
    res = data['results']
    params = data['parameters']
    N = len(res)
    
    print(f"================ 文件: {os.path.basename(filename)} ================")
    md_table = f"### {params.get('noise_type', 'normal')} 噪声下的性能对比 (Ranking, N={N})\n"
    md_table += "| Method | RMSE (95% CI) | MAE | Accuracy | Time (s) |\n"
    md_table += "|---|---|---|---|---|\n"
    
    methods = ['Pooled', 'Local', 'Avg', 'D-subGD', 'Proposed']
    
    avg_hist = None
    avg_avg_rmse = None
    
    for method in methods:
        # 检查该方法是否在结果中存在 (用户可能设置为 False 跳过了)
        if method in res[0]:
            rmses = [r[method]['RMSE'] for r in res]
            maes = [r[method]['MAE'] for r in res]
            accs = [r[method]['Accuracy'] for r in res]
            times = [r[method]['Time'] for r in res]
            
            mean_rmse = np.mean(rmses)
            # 计算 95% 置信区间: 1.96 * std / sqrt(N)
            ci_rmse = 1.96 * np.std(rmses) / np.sqrt(N)
            mean_mae = np.mean(maes)
            mean_acc = np.mean(accs)
            mean_time = np.mean(times)
            
            md_table += f"| {method} | {mean_rmse:.4f} ± {ci_rmse:.4f} | {mean_mae:.4f} | {mean_acc:.2%} | {mean_time:.2f} |\n"
            
            if method == 'Proposed' and 'hist_rmse' in res[0]['Proposed']:
                avg_hist = np.mean([r['Proposed']['hist_rmse'] for r in res], axis=0)
            if method == 'Avg':
                avg_avg_rmse = mean_rmse
                
    print(md_table)
    print("====================================================================\n")
    
    if avg_hist is not None:
        plt.figure(figsize=(8, 5))
        plt.plot(avg_hist, marker='o', label='Proposed RMSE (Avg)')
        if avg_avg_rmse is not None:
            plt.axhline(avg_avg_rmse, color='r', linestyle='--', label='Avg MR RMSE')
        plt.xlabel('Outer Iteration t')
        plt.ylabel('RMSE')
        plt.title(f"Convergence of Proposed (Ranking) - {params.get('noise_type', 'normal')} noise")
        plt.legend()
        plt.grid(True)
        plt.show()
